# Gain chains: products in logarithmic coordinates

An application of Alpha Calculus with explicit steps, code and limitations.

This notebook is self-contained. It requires logarithms and basic Python.
By the end, you should be able to explain multiplicative contributions,
update a known stage, and recognise range and precision limits.

1. Choose coordinates for multiplicative composition.
2. Replace one stage and compare three calculation methods.
3. Count the work required for repeated queries.
4. Check extreme ranges and cancellation during successive updates.

Run all cells in order in a fresh runtime. Only the standard library is used:
no other notebook, book, network connection, private data or Drive access is
required. Tables are exported as CSV files to `results/` beside the notebook.


In [1]:
import csv
import json
import math
import platform
import random
from fractions import Fraction
from pathlib import Path

RESULTS = Path("results")
RESULTS.mkdir(parents=True, exist_ok=True)

print("Python:", platform.python_version())
print("Float precision: approximately 16 significant digits.")

Python: 3.14.3
Float precision: approximately 16 significant digits.


## 1. Recognising a useful structure

A generator $α$ and its inverse allow operations to be expressed in different
coordinates. With $α(t)=e^t$, from $ℝ$ to $(0,∞)$, transported addition is

$$x\oplus_\alpha y
=\exp(\log x+\log y)=xy.$$

The Alpha additive identity is **1**, because $\log1=0$. An attenuation
$0<x<1$ has a negative coordinate. Zero and negative values are excluded
for this choice of generator.

This equality follows from Definitions 1.1.1 and 1.1.5 and Example 1.1.3
of *Alpha Calculus*, **printed pp. 1–2 / PDF pp. 10–11**.
The following examples are educational examples developed for this laboratory.


In [2]:
def positive_log(value):
    """Coordinate of a positive, finite, dimensionless quantity."""
    if not math.isfinite(value) or value <= 0:
        raise ValueError("The quantity must be positive and finite.")
    return math.log(value)


def product_from_log(total):
    """Recover a positive product representable as a float."""
    if not math.isfinite(total):
        raise ValueError("The logarithmic sum must be finite.")
    product = math.exp(total)
    if product == 0.0:
        raise ArithmeticError("The positive product was rounded to zero.")
    return product

In [3]:
def print_table(
        title,
        header,
        rows,
    ):
    """Print a small table with aligned columns."""
    data = [
        [str(value) for value in row]
        for row in [header, *rows]
        ]
    widths = [
        max(map(len, column))
        for column in zip(*data)
        ]

    print(title)
    for index, row in enumerate(data):
        print(
                " | ".join(
                        value.ljust(width)
                        for value, width in zip(row, widths)
                    )
            )
        if index == 0:
            print("-+-".join("-" * width for width in widths))


def export_table(
        name,
        header,
        rows,
    ):
    """Save table data as UTF-8 CSV."""
    with (RESULTS / name).open(
            "w",
            newline="",
            encoding="utf-8",
        ) as file:
        writer = csv.writer(file)
        writer.writerow(header)
        writer.writerows(rows)


## 2. Application: changing one stage in a gain chain

### Situation and recognition cues

A chain of stages applies dimensionless gains $g_i>0$.
Their effects **multiply**:

$$G=\prod_{i=1}^n g_i.$$

The task is to replace a known stage repeatedly and query the total.
The useful cue is the combination of **multiplicative composition** and
**local changes**. Each gain is a dimensionless multiplicative factor.

**Assumptions:** positive, finite factors; a known index; and multiplicative
composition appropriate to the problem. If factors depend on the state of
other stages, that dependence must be modelled before applying this procedure.

### Step 1. Making the contributions explicit

For $g=[2,3,5]$, store $b_i=\log g_i$ and
$S=\sum_i b_i$. The product is $e^S=30$.


In [4]:
gains = [2.0, 3.0, 5.0]
coordinates = [positive_log(value) for value in gains]
total_log = math.fsum(coordinates)

stage_rows = [
    [index + 1, gain, f"{coordinate:.9f}"]
    for index, (gain, coordinate)
    in enumerate(zip(gains, coordinates))
    ]
print_table(
        "Contribution by stage",
        ["Stage", "g", "log(g)"],
        stage_rows,
    )
print(f"S = {total_log:.9f}; exp(S) = {math.exp(total_log):.9f}")

assert math.isclose(math.exp(total_log), 30.0, rel_tol=1e-12)
export_table(
        "gain_stages.csv",
        ["stage", "g", "log_g"],
        stage_rows,
    )

Contribution by stage
Stage | g   | log(g)     
------+-----+------------
1     | 2.0 | 0.693147181
2     | 3.0 | 1.098612289
3     | 5.0 | 1.609437912
S = 3.401197382; exp(S) = 30.000000000


### Step 2. Replacing $3\to4$

Remove the previous logarithm and add the new one:

$$S'=S-\log3+\log4=\log40.$$

The increment $\Delta S=\log(4/3)$ explains the relative change:
$G'/G=4/3$. Compare three methods that give the same result.


In [5]:
index = 1
new_gain = 4.0
new_gains = gains.copy()
new_gains[index] = new_gain

new_total = math.fsum(
        [
            total_log,
            -coordinates[index],
            positive_log(new_gain),
            ]
        )
via_log = product_from_log(new_total)
via_quotient = math.prod(gains) / gains[index] * new_gain
via_recomputation = math.prod(new_gains)

print_table(
        "Three methods, one replacement",
        ["Method", "New product"],
        [
            ["Recompute", f"{via_recomputation:.12g}"],
            ["Product/quotient", f"{via_quotient:.12g}"],
            ["Sum of logarithms", f"{via_log:.12g}"],
            ],
        )
assert math.isclose(via_log, 40.0, rel_tol=1e-12)
assert via_quotient == via_recomputation == 40.0

Three methods, one replacement
Method            | New product
------------------+------------
Recompute         | 40         
Product/quotient  | 40         
Sum of logarithms | 40         


### Step 3. Counting the relevant work

For $n$ stages and $k$ replacements at known indices, each followed by a
query of the total, assume fixed precision and unit-cost elementary
operations. Preparing the state costs $O(n)$ for all three methods.

| Method | Work per replacement and query | Total |
|---|---|---|
| Recompute the product | $O(n)$ | $O(n+kn)$ |
| Maintain product/quotient state | $O(1)$ | $O(n+k)$ |
| Maintain a sum of logarithms | $O(1)$ | $O(n+k)$ |

The logarithmic method requires a new logarithm, a three-term sum, and an
exponential when the product is requested. `fsum` is not equivalent to a
single machine addition. Storing the factors or coordinates uses $O(n)$ memory.

**Alpha's contribution here:** a coherent guide to choosing a representation
and explaining contributions. The asymptotic improvement comes from maintaining
state; product/quotient has the same order of cost. A conventional method
that maintains logarithms is the same algorithm. No timings are measured.


### Step 4. Checking the workload without claiming a universal guarantee

Repeat 256 replacements over 16 gains, using a fixed seed.
The gains are **powers of two**: the product/quotient baseline agrees exactly
with the rational reference for this workload. That exactness does not extend
to arbitrary floating-point factors.


In [6]:
rng = random.Random(975)
factors = [2.0 ** rng.randint(-3, 3) for _ in range(16)]
logs = [math.log(value) for value in factors]
log_sum = math.fsum(logs)
product_quotient = math.prod(factors)
max_error = 0.0

for _ in range(256):
    position = rng.randrange(len(factors))
    new_value = 2.0 ** rng.randint(-3, 3)
    new_log = math.log(new_value)
    log_sum = math.fsum([log_sum, -logs[position], new_log])
    product_quotient = product_quotient / factors[position] * new_value
    factors[position] = new_value
    logs[position] = new_log
    reference = float(math.prod(Fraction(value) for value in factors))
    computed = product_from_log(log_sum)
    assert product_quotient == reference
    assert math.isclose(computed, reference, rel_tol=1e-12)
    max_error = max(max_error, abs(computed / reference - 1))

print("Replacements checked: 256 / 256")
print(f"Maximum relative error of the log method: {max_error:.3e}")
print("Product/quotient: exact for this powers-of-two workload.")

Replacements checked: 256 / 256
Maximum relative error of the log method: 2.887e-15
Product/quotient: exact for this powers-of-two workload.


### Step 5. Preserving range: a representable logarithm need not imply a representable product

The product $10^{200}10^{200}=10^{400}$ is outside the range of the usual
`float`, although its logarithm $400\log10$ is representable. For
$10^{-400}$, underflow occurs: the computed zero is not the real product.

For the chain $[10^{200},10^{200},10^{-200},10^{-200}]$, the real result
is 1. Sequential multiplication can overflow before the factors balance.
Reordering the factors is a useful conventional baseline in this example.


In [7]:
def describe_return(log_sum):
    try:
        return f"{product_from_log(log_sum):.9g}"
    except OverflowError:
        return "Overflow"
    except ArithmeticError:
        return "Underflow to zero"


range_rows = []
for label, values in [
        ("Large", [1e200, 1e200]),
        ("Small", [1e-200, 1e-200]),
        ("Balanced", [1e200, 1e200, 1e-200, 1e-200]),
        ]:
    range_sum = math.fsum(math.log(value) for value in values)
    range_rows.append(
            [
                label,
                math.prod(values),
                f"{range_sum:.6f}",
                describe_return(range_sum),
                ]
            )

print_table(
        "Representable range",
        ["Case", "Direct product", "S", "exp(S)"],
        range_rows,
        )
export_table(
        "gain_range.csv",
        ["case", "direct", "log_total", "return"],
        range_rows,
        )
print("Product with reordered factors:", math.prod([1e200, 1e-200] * 2))
assert describe_return(0.0) == "1"

Representable range
Case     | Direct product | S           | exp(S)           
---------+----------------+-------------+------------------
Large    | inf            | 921.034037  | Overflow         
Small    | 0.0            | -921.034037 | Underflow to zero
Balanced | inf            | 0.000000    | 1                
Product with reordered factors: 1.0


### Limitation: `fsum` cannot recover information already lost

Start with coordinates $[700,-700,\log(1+\delta)]$, where
$1+\delta=1.00000000000001$. Replace the first two gains with 1.
In exact real arithmetic, the small third contribution remains. A
floating-point accumulator can lose it during the updates, even with `fsum`.

Recomputing from the components recovers that contribution in this example;
it costs $O(n)$ and must be counted if a rebuilding policy is adopted.
Neither the counterexample nor the repair provides a general error bound.


In [8]:
small_factor = 1.00000000000001
small_term = math.log(small_factor)
accumulated = math.fsum([700.0, -700.0, small_term])
accumulated = math.fsum([accumulated, -700.0, 0.0])
accumulated = math.fsum([accumulated, 700.0, 0.0])
rebuilt = math.fsum([0.0, 0.0, small_term])

print_table(
        "Cancellation during successive updates",
        ["Method", "Product"],
        [
            ["Expected", f"{small_factor:.17g}"],
            ["Incremental accumulator", f"{math.exp(accumulated):.17g}"],
            ["Reconstruction", f"{math.exp(rebuilt):.17g}"],
            ],
        )
assert accumulated == 0.0
assert math.exp(accumulated) != small_factor
assert math.exp(rebuilt) == small_factor

Cancellation during successive updates
Method                  | Product         
------------------------+-----------------
Expected                | 1.00000000000001
Incremental accumulator | 1               
Reconstruction          | 1.00000000000001


### Exercise

The chain is $[0.5,2,8]$. Change the second gain to 3.
Before running the next cell, find the new product. Which logarithmic
contribution is removed, and which is added? Why does attenuation not
require a complex logarithm?

The next cell provides the solution for comparison.


In [9]:
exercise = [0.5, 2.0, 8.0]
answer_log = math.fsum(
        [
            math.fsum(math.log(value) for value in exercise),
            -math.log(2.0),
            math.log(3.0),
            ]
        )
print("Answer:", math.exp(answer_log))
assert math.isclose(math.exp(answer_log), 12.0, rel_tol=1e-12)

Answer: 12.0


## 3. When this representation is useful

The sum of logarithms makes multiplicative contributions explicit and
preserves the logarithmic total even when the product is outside the range of
`float`. Maintaining state avoids recomputing the entire chain after each change.

Product/quotient achieves the same order of cost. A conventional method
that maintains logarithms is the same algorithm. The asymptotic improvement
comes from the maintained state; no novelty or exclusive Alpha advantage is
claimed. Before choosing an implementation, specify the required range,
precision and rebuilding frequency: an accurate sum of three operands cannot
recover information already lost in an earlier total.

**Proposed extension:** investigate a relevant public workload, a rebuilding
policy and precision requirements before measuring timings or turning the
demonstration into a production API.

## Sources and scope

- Georgiev and Aydin, *Alpha Calculus*, 1st ed., 2026,
  [DOI 10.1201/9781003750734](https://doi.org/10.1201/9781003750734).
  Passages checked: **printed pp. 1–2 / PDF pp. 10–11**, Definitions
  1.1.1 and 1.1.5 and Example 1.1.3: exponential generator and transported addition.
- [Python: `math`](https://docs.python.org/3.14/library/math.html),
  `log`, `exp`, `fsum`, `prod` and `isclose`: operations and implementation
  limits. Range and cancellation are checked in this notebook.

The application, examples and code are educational material developed for this
laboratory, building on an earlier local demonstration. They are not presented
as a case from the book or as a scientifically novel algorithm. The 256
replacements use powers of two and do not establish bounds for arbitrary
floats. The book is not distributed. The checks cover the cited passages
and examples only.


In [10]:
summary = {
    "status": "passed",
    "application": "gain_chains",
    "python": platform.python_version(),
    "replacements": 256,
    "factors": 16,
    "seed": 975,
    "workload": "Powers of two, integer exponents from -3 to 3.",
    "max_log_relative_error": max_error,
    "example_30_to_40": via_log,
    "incremental_cancellation": math.exp(accumulated),
    "rebuilt_cancellation": math.exp(rebuilt),
    }
(RESULTS / "checks.json").write_text(
        json.dumps(summary, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
        )
print("Checks passed. CSV files and summary in:", RESULTS)

Checks passed. CSV files and summary in: results
